# MLflow cho LLM, Prompt Engineering và AI Agent

Notebook này dành cho lớp ứng dụng Generative AI, nơi MLflow không chỉ dùng để log model mà còn dùng để:
- trace prompt, response, tool call và latency,
- quản lý prompt version,
- đánh giá chất lượng đầu ra bằng judge/scorer,
- theo dõi workflow agent hoặc multi-step chain.

Phần này khác rõ rệt so với ML và Deep Learning truyền thống, vì đối tượng cần quản lý không chỉ là weight của model, mà còn là hành vi của cả ứng dụng GenAI.

## 1. LLM, RAG và AI Agent khác nhau thế nào?

Phân biệt ngắn gọn:
- `LLM application`: gửi prompt vào model và nhận câu trả lời.
- `RAG application`: ngoài LLM còn có bước truy xuất tài liệu rồi mới sinh câu trả lời.
- `AI Agent`: ngoài việc sinh câu trả lời, hệ thống còn biết chọn công cụ, lên kế hoạch, gọi API, lặp nhiều bước và sử dụng trạng thái trung gian.

Các pattern kiến trúc phổ biến:
- `Direct prompt -> response`: ứng dụng chat hoặc QA đơn giản.
- `Retriever -> prompt -> response`: RAG cơ bản.
- `Planner -> tools -> synthesizer`: agent có lập kế hoạch.
- `Router -> specialist agents`: multi-agent hoặc delegated workflow.

Khi hệ thống nhiều bước hơn, MLflow tracing trở nên quan trọng hơn hẳn tracking kiểu cũ.

## 2. MLflow hỗ trợ gì cho GenAI?

Theo tài liệu MLflow mới, nhóm tính năng quan trọng cho GenAI gồm:
- `Tracing / Observability`: theo dõi toàn bộ execution của ứng dụng.
- `Prompt Registry`: version prompt giống như version artifact.
- `Evaluation`: chấm chất lượng đầu ra bằng judge hoặc scorer.
- `Framework integrations`: OpenAI, LangChain, LangGraph và nhiều framework khác.

Với GenAI, bạn nên ưu tiên nghĩ theo 4 lớp:
1. Prompt và model configuration.
2. Trace của từng request.
3. Evaluation dataset và judge/scorer.
4. Versioning của app hoặc agent.


In [ ]:
# Gợi ý cài thư viện cho workflow GenAI với MLflow.
# Phần serve model chạy riêng bằng vLLM, notebook này chỉ gọi endpoint OpenAI-compatible.
# %pip install -q "mlflow[genai]" openai langchain langchain-openai langgraph pandas


## 3. Khởi động MLflow server và vLLM endpoint

Workflow demo này cần 2 process chạy song song:
- `mlflow server` để xem tracing, prompt registry và evaluation.
- `vllm serve` để host `Qwen/Qwen3.5-0.8B` qua OpenAI-compatible API.

Ví dụ local cho MLflow:

```bash
cd /home/pytorch/DataScience/MLOps-VNPost
mlflow server --host 127.0.0.1 --port 5000
```

Ví dụ local cho vLLM:

```bash
vllm serve Qwen/Qwen3.5-0.8B \
  --port 8000 \
  --tensor-parallel-size 1 \
  --max-model-len 32768 \
  --enable-auto-tool-choice \
  --tool-call-parser qwen3_coder \
  --language-model-only
```

Các cell bên dưới mặc định dùng:
- `MLFLOW_TRACKING_URI=http://127.0.0.1:5000`
- `OPENAI_BASE_URL=http://127.0.0.1:8000/v1`
- `OPENAI_API_KEY=EMPTY`


In [ ]:
import os
import pandas as pd
import mlflow
from openai import OpenAI

tracking_uri = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
vllm_base_url = os.getenv("OPENAI_BASE_URL", "http://127.0.0.1:8000/v1")
vllm_model = os.getenv("OPENAI_MODEL", "Qwen/Qwen3.5-0.8B")
openai_api_key = os.getenv("OPENAI_API_KEY", "EMPTY")

os.environ["OPENAI_BASE_URL"] = vllm_base_url
os.environ["OPENAI_API_BASE"] = vllm_base_url
os.environ["OPENAI_API_KEY"] = openai_api_key

mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("mlflow_llm_agent_qwen_vllm")

client = OpenAI(base_url=vllm_base_url, api_key=openai_api_key)

def assert_vllm_ready() -> list[str]:
    try:
        models = [model.id for model in client.models.list().data]
    except Exception as exc:
        raise RuntimeError(
            f"Không kết nối được tới vLLM tại {vllm_base_url}. Hãy start server trước khi chạy các cell tiếp theo."
        ) from exc

    print("Tracking URI:", tracking_uri)
    print("vLLM Base URL:", vllm_base_url)
    print("Served models:", models)
    return models

available_models = assert_vllm_ready()
if vllm_model not in available_models:
    print(f"Warning: {vllm_model} chưa xuất hiện trong /models. Notebook vẫn sẽ dùng model_id này.")


## 4. Trace một lời gọi LLM trực tiếp qua vLLM

Nếu ứng dụng của bạn chỉ gọi model trực tiếp, bước đầu tiên nên làm là bật autolog/tracing cho client SDK. Ví dụ dưới đây vẫn dùng OpenAI Python client, nhưng `base_url` trỏ vào endpoint local của `vLLM`.

Mục tiêu của ví dụ:
- ghi lại prompt và response,
- theo dõi độ trễ và metadata request,
- tạo trace đầu tiên trong MLflow UI cho model self-hosted.


In [ ]:
mlflow.openai.autolog()

with mlflow.start_run(run_name="direct_llm_call_qwen_vllm"):
    response = client.chat.completions.create(
        model=vllm_model,
        messages=[
            {"role": "system", "content": "Bạn là trợ lý MLOps, trả lời ngắn gọn bằng tiếng Việt."},
            {"role": "user", "content": "Giải thích vì sao experiment tracking quan trọng trong MLOps."},
        ],
        temperature=0.3,
        max_tokens=256,
        extra_body={"top_k": 20},
    )

answer = response.choices[0].message.content
print(answer)


## 5. Quản lý prompt bằng Prompt Registry

Trong ứng dụng LLM thật, prompt thường thay đổi liên tục. Nếu chỉ để prompt trong source code, bạn rất khó:
- biết phiên bản nào đang chạy trong production,
- so sánh prompt cũ và mới,
- tái hiện đúng cấu hình của một lần đánh giá.

Prompt Registry giải quyết bài toán đó bằng cách version prompt tương tự version model hoặc artifact. Trong demo này, ta lưu luôn model ID và cấu hình endpoint `vLLM`.


In [ ]:
prompt = mlflow.genai.register_prompt(
    name="qa-assistant-qwen-vllm",
    template=(
        "Trả lời câu hỏi dựa trên ngữ cảnh.\n\n"
        "Ngữ cảnh: {{context}}\n\n"
        "Câu hỏi: {{question}}\n\n"
        "Yêu cầu: trả lời ngắn gọn bằng tiếng Việt."
    ),
    commit_message="Prompt đầu tiên cho demo Qwen3.5-0.8B qua vLLM",
    model_config={
        "model_name": vllm_model,
        "endpoint_type": "openai-compatible",
        "base_url": vllm_base_url,
        "temperature": 0.3,
        "max_tokens": 256,
        "extra_body": {"top_k": 20},
    },
)

latest_prompt = mlflow.genai.load_prompt("prompts:/qa-assistant-qwen-vllm@latest")
rendered = latest_prompt.format(
    context="MLflow lưu parameters, metrics, artifacts, prompt versions và traces.",
    question="MLflow giúp gì cho ứng dụng GenAI?",
)
print(rendered)


## 6. Đánh giá đầu ra bằng `mlflow.genai.evaluate()`

Tracking trace là chưa đủ. Với GenAI, bạn còn cần biết chất lượng đầu ra có cải thiện hay không khi đổi prompt, đổi model hoặc đổi logic agent.

MLflow cho phép đánh giá bằng:
- built-in judge như `Correctness`, `Guidelines`,
- custom scorer do bạn tự viết,
- tập dữ liệu evaluation riêng của team.

Để demo local với `Qwen/Qwen3.5-0.8B`, ví dụ dưới đây chỉ dùng `predict_fn` và custom scorers, nên không cần thêm một judge model bên ngoài.


In [ ]:
from mlflow.entities import Feedback
from mlflow.genai import scorer

eval_dataset = [
    {
        "inputs": {"question": "MLflow lưu những gì trong một run?"},
        "expectations": {"required_keywords": ["tham số", "metrics", "artifacts"]},
    },
    {
        "inputs": {"question": "Vì sao cần experiment tracking trong MLOps?"},
        "expectations": {"required_keywords": ["so sánh", "tái lập"]},
    },
]

@mlflow.trace(name="qwen_eval_predict")
def qa_predict_fn(question: str) -> str:
    response = client.chat.completions.create(
        model=vllm_model,
        messages=[
            {
                "role": "system",
                "content": "Trả lời ngắn gọn bằng tiếng Việt và giữ lại các thuật ngữ kỹ thuật quan trọng khi cần.",
            },
            {"role": "user", "content": question},
        ],
        temperature=0.2,
        max_tokens=160,
        extra_body={"top_k": 20},
    )
    return response.choices[0].message.content

@scorer
def keyword_coverage(*, outputs: str, expectations: dict) -> Feedback:
    missing = [
        keyword for keyword in expectations["required_keywords"] if keyword.lower() not in outputs.lower()
    ]
    passed = len(missing) == 0
    rationale = "Đủ từ khóa kỳ vọng." if passed else f"Thiếu từ khóa: {missing}"
    return Feedback(value=passed, rationale=rationale)

@scorer
def is_brief(*, outputs: str) -> bool:
    return len(outputs.split()) <= 60

scorers = [keyword_coverage, is_brief]

results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=qa_predict_fn,
    scorers=scorers,
)

results.tables["eval_results_table"].head()


## 7. Trace một AI Agent bằng LangChain / LangGraph

Theo tài liệu MLflow hiện tại, `mlflow.langchain.autolog()` cũng được dùng để trace LangChain và LangGraph. Điều này đặc biệt hữu ích với agent vì lúc này bạn có thể quan sát:
- bước suy luận trung gian,
- tool nào được gọi,
- input/output của từng node,
- nơi nào agent bị lặp hoặc sai logic.

Nếu muốn demo tool calling với `Qwen/Qwen3.5-0.8B`, hãy bảo đảm `vllm serve` đã bật `--enable-auto-tool-choice` và `--tool-call-parser qwen3_coder`.


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

@tool
def get_tracking_best_practice(topic: str) -> str:
    """Trả về một best practice ngắn cho experiment tracking."""
    practices = {
        "production": "Log prompt, response, latency và tool calls cho từng request production.",
        "prompt": "Version prompt cùng model config để tái hiện đúng hành vi.",
        "evaluation": "Dùng evaluation dataset cố định để so sánh agent qua nhiều phiên bản.",
    }
    return practices.get(
        topic.lower(),
        "Luôn log parameters, metrics, artifacts và prompt versions cho mỗi phiên bản ứng dụng.",
    )

mlflow.langchain.autolog()

agent_model = ChatOpenAI(
    model=vllm_model,
    temperature=0,
    openai_api_base=vllm_base_url,
    openai_api_key=openai_api_key,
)
agent = create_react_agent(agent_model, tools=[get_tracking_best_practice])

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Tôi đang demo MLflow cho agent. Hãy dùng tool phù hợp và đưa ra một best practice cho production."
                ),
            }
        ]
    }
)

print(result["messages"][-1].content)


## 8. Cách chia file trong repo này

Để tránh lẫn phạm vi, bộ tài liệu MLflow trong repo nên hiểu như sau:
- `mlflow_ML.ipynb`: classical ML, benchmark thuật toán, model artifact.
- `mlflow_DeepLearning.ipynb`: PyTorch, Transformer, Hugging Face.
- `mlflow_LLM_Agent.ipynb`: tracing, prompt registry, evaluation, agent workflow.

Đây là cách tách hợp lý hơn việc dồn tất cả vào một notebook duy nhất.

Tài liệu chính thức nên đọc thêm:
- GenAI overview: `https://mlflow.org/docs/latest/genai/`
- Evaluation quickstart: `https://mlflow.org/docs/latest/genai/eval-monitor/quickstart/`
- Prompt Registry: `https://mlflow.org/docs/latest/genai/prompt-registry/`
- LangChain autologging: `https://mlflow.org/docs/latest/genai/flavors/langchain/autologging/`
- LangGraph tracing: `https://mlflow.org/docs/latest/genai/tracing/integrations/listing/langgraph`
- Qwen3.5 model card và ví dụ `vllm serve`: `https://huggingface.co/Qwen/Qwen3.5-0.8B`
